In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
import os

In [37]:
Z_DIM = 32 #latent space
HIDDEN_DIM = 128 #lstm hidden state
ACTION_DIM = 2 #steering+throttle
NUM_GAUSSIANS = 5 #for mixture density
BATCH_SIZE = 32
IMG_SIZE = 64
SEQ_LEN = 30 #frames per training seq
NUM_SEQUENCES = 5000 #total seqs (5000*30=15000 total frames)
SEED=42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [38]:
DEVICE

device(type='cuda')

In [39]:
# frame (car position, lane phase , object depth, object lane)
def render_frame(car_pos=0.0, lane_phase=0.0, obj_z=0.0, obj_lane=0.0, img_size=IMG_SIZE):
    frame = np.zeros((img_size, img_size, 3), dtype=np.uint8) #empty black array
    frame[:img_size // 2, :] = [135, 206, 235]  # Sky - top half
    frame[img_size // 2:, :] = [34, 139, 34]   # Grass - bottom half

    # Road trapezoid
    road_pts = np.array([
        [img_size // 2 - 2, img_size // 2],
        [img_size // 2 + 2, img_size // 2],
        [img_size + 40, img_size],
        [-40, img_size]
    ])
    road_pts[:, 0] -= int(car_pos * 40) # shift road horizontally by car pos * 40pixels
    cv2.fillPoly(frame, [road_pts], (100, 100, 100))

    # Lane dashes
    for i in range(5):
        z = (i + lane_phase) / 5
        y = int(img_size // 2 + z * (img_size // 2))
        x_center = img_size // 2 - int(car_pos * 40 * z)
        if y < img_size:
            cv2.line(frame, (x_center, y), (x_center, min(y + int(5*z), img_size)), (255, 255, 255), max(1, int(2*z)))

    # Moving object (to show the moving car)
    # depends on lane and car position
    if obj_z > 0:
        y = int(img_size // 2 + obj_z * (img_size // 2))
        x_center = img_size // 2 + int((obj_lane - car_pos) * 40 * obj_z)
        w = int(2 + obj_z * 20)
        h = int(1 + obj_z * 10)
        if y < img_size:
            cv2.rectangle(frame, (x_center - w//2, y - h), (x_center + w//2, y), (200, 50, 50), -1)

    return frame.transpose(2, 0, 1) / 255.0 #normalize

In [40]:
class DrivingSimulationDataset(Dataset):
    def __init__(self, num_sequences=NUM_SEQUENCES, seq_len=SEQ_LEN, img_size=IMG_SIZE):
        self.num_sequences = num_sequences
        self.seq_len = seq_len
        self.img_size = img_size
#synthetic gen
    def __len__(self):
        return self.num_sequences

    def __getitem__(self, idx): #set defaults
        car_pos = 0.0
        lane_phase = 0.0
        obj_z = 0.0
        obj_lane = np.random.choice([-0.4, 0.4])
        obj_speed = np.random.uniform(0.02, 0.05)

        frames = []
        actions = []
        for _ in range(self.seq_len): #generate new frame and add to empty list
            frames.append(render_frame(car_pos=car_pos, lane_phase=lane_phase, obj_z=obj_z, obj_lane=obj_lane, img_size=self.img_size))
            action = np.random.uniform(-0.15, 0.15, size=2) #random action and add to action list
            actions.append(action)
            car_pos += action[0] #update car pos by steering value
            car_pos = np.clip(car_pos, -1.0, 1.0) #clamp so car doesnt run out of the lane
            lane_phase = (lane_phase + 0.1) % 1.0
            obj_z += obj_speed #increase depth as we move
            if obj_z > 1.2: #reset (when car passes, reset with new random lane)
                obj_z = 0.0
                obj_lane = np.random.choice([-0.4, 0.4]) #random lane selection here after reset
        return torch.tensor(np.array(frames), dtype=torch.float32), torch.tensor(np.array(actions), dtype=torch.float32)

In [41]:
class VAE(nn.Module): #basic vae to work with frames and use the latent space
    def __init__(self, z_dim=32):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(256 * 4 * 4, z_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, z_dim)
        self.decoder_fc = nn.Linear(z_dim, 256 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (256, 4, 4)),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1), nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decoder(self.decoder_fc(z))
        return recon_x, z, mu, logvar

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        return self.decoder(self.decoder_fc(z))

In [42]:
class MDNRNN(nn.Module): #with lstm
    def __init__(self, z_dim=32, action_dim=2, hidden_dim=128, num_gaussians=5):
        super(MDNRNN, self).__init__() #args
        self.z_dim = z_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.num_gaussians = num_gaussians
        self.rnn = nn.LSTM(z_dim + action_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_gaussians * z_dim * 3)

    def forward(self, z, action, hidden=None): #concat current z and action with last feature dim
        #pass it through the lstm, fully connect and reshape
        x = torch.cat([z, action], dim=-1)
        output, hidden = self.rnn(x, hidden)
        params = self.fc(output)
        bs, seq_len, _ = output.shape
        params = params.view(bs, seq_len, self.num_gaussians, 3, self.z_dim)
        pi = F.softmax(params[:, :, :, 0, 0], dim=2) #get mean predictions,
        mu = params[:, :, :, 1, :]
        sigma = torch.exp(params[:, :, :, 2, :]) #exp to to std preds to get positive values
        return pi, mu, sigma, hidden

In [43]:
def vae_loss(recon_x, x, mu, logvar): #with kl divergance
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

def mdn_loss(pi, mu, sigma, target):
    # added as it exploded for 30 epochs (nan clips)
    mu = torch.nan_to_num(mu, nan=0.0, posinf=0.0, neginf=0.0)
    sigma = torch.nan_to_num(sigma, nan=1.0, posinf=10.0, neginf=0.1)
    sigma = torch.clamp(sigma, min=0.01, max=10.0)
    # create normal dist
    target = target.unsqueeze(2).expand_as(mu)
    m = torch.distributions.Normal(mu, sigma)
    log_probs = m.log_prob(target).sum(dim=-1)
    log_pi = torch.log(pi + 1e-8)
    log_pi = torch.nan_to_num(log_pi, nan=-20.0)
    loss = -torch.logsumexp(log_pi + log_probs, dim=-1)
    loss = torch.nan_to_num(loss, nan=0.0)
    return loss.mean()

In [44]:
def train_vae(vae, dataloader, device=DEVICE, epochs=30):
    optimizer = optim.Adam(vae.parameters(), lr=1e-3)
    vae.to(device)
    vae.train()
    for epoch in range(epochs):
        total_loss = 0
        total_items = 0
        for frames, _ in dataloader:
            frames = frames.view(-1, 3, 64, 64).to(device)
            optimizer.zero_grad()
            recon_x, z, mu, logvar = vae(frames)
            loss = vae_loss(recon_x, frames, mu, logvar)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            total_items += frames.size(0)
        avg_loss = total_loss / total_items if total_items > 0 else 0
        print(f'Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}')

In [45]:
def train_rnn(vae, rnn, dataloader, device=DEVICE, epochs=30):
    optimizer = optim.Adam(rnn.parameters(), lr=1e-3)
    vae.to(device)
    vae.eval()
    rnn.to(device)
    rnn.train()
    for epoch in range(epochs):
        total_loss = 0
        total_items = 0
        for frames, actions in dataloader:
            frames, actions = frames.to(device), actions.to(device)
            with torch.no_grad():
                bs, seq_len, c, h, w = frames.shape #split seqs
                flat_frames = frames.view(-1, c, h, w)
                mu_z, _ = vae.encode(flat_frames)
                z = mu_z.view(bs, seq_len, -1)
            z_input = z[:, :-1, :]
            actions_input = actions[:, :-1, :]
            z_target = z[:, 1:, :]
            optimizer.zero_grad()
            pi, mu_pred, sigma_pred, _ = rnn(z_input, actions_input)
            # skip batch if preds diverge
            if torch.isnan(mu_pred).any() or torch.isnan(sigma_pred).any():
                print('skipped batch: nan in preds')
                continue
            loss = mdn_loss(pi, mu_pred, sigma_pred, z_target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=5.0)            
            optimizer.step()
            total_loss += loss.item()
            total_items += z_input.size(0)
        avg_loss = total_loss / total_items if total_items > 0 else 0
        print(f'Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}')

In [46]:
dataset = DrivingSimulationDataset(num_sequences=NUM_SEQUENCES, seq_len=SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

vae = VAE(z_dim=Z_DIM).to(DEVICE)
rnn = MDNRNN(z_dim=Z_DIM, action_dim=ACTION_DIM, hidden_dim=HIDDEN_DIM, num_gaussians=NUM_GAUSSIANS).to(DEVICE)

In [47]:
train_vae(vae, dataloader, DEVICE, epochs=30)

Epoch 1, Avg Loss: 7099.7719
Epoch 2, Avg Loss: 6919.3538
Epoch 3, Avg Loss: 6892.1763
Epoch 4, Avg Loss: 6858.3363
Epoch 5, Avg Loss: 6852.5011
Epoch 6, Avg Loss: 6848.1641
Epoch 7, Avg Loss: 6846.3818
Epoch 8, Avg Loss: 6845.0427
Epoch 9, Avg Loss: 6839.5558
Epoch 10, Avg Loss: 6829.4723
Epoch 11, Avg Loss: 6826.8678
Epoch 12, Avg Loss: 6825.4102
Epoch 13, Avg Loss: 6822.5753
Epoch 14, Avg Loss: 6821.7199
Epoch 15, Avg Loss: 6820.5510
Epoch 16, Avg Loss: 6820.1937
Epoch 17, Avg Loss: 6819.5488
Epoch 18, Avg Loss: 6818.1301
Epoch 19, Avg Loss: 6817.9848
Epoch 20, Avg Loss: 6816.5539
Epoch 21, Avg Loss: 6815.4967
Epoch 22, Avg Loss: 6814.0176
Epoch 23, Avg Loss: 6814.6431
Epoch 24, Avg Loss: 6813.3161
Epoch 25, Avg Loss: 6810.9956
Epoch 26, Avg Loss: 6811.4004
Epoch 27, Avg Loss: 6810.9552
Epoch 28, Avg Loss: 6810.0042
Epoch 29, Avg Loss: 6808.6158
Epoch 30, Avg Loss: 6807.2668


In [48]:
train_rnn(vae, rnn, dataloader, DEVICE, epochs=30)

Epoch 1, Avg Loss: -0.7517
Epoch 2, Avg Loss: -1.3055
Epoch 3, Avg Loss: -1.4025
Epoch 4, Avg Loss: -1.4816
Epoch 5, Avg Loss: -1.5635
Epoch 6, Avg Loss: -1.6541
Epoch 7, Avg Loss: -1.7334
Epoch 8, Avg Loss: -1.7872
Epoch 9, Avg Loss: -1.8201
Epoch 10, Avg Loss: -1.8570
Epoch 11, Avg Loss: -1.8848
Epoch 12, Avg Loss: -1.9145
Epoch 13, Avg Loss: -1.9382
Epoch 14, Avg Loss: -1.9553
Epoch 15, Avg Loss: -1.9663
Epoch 16, Avg Loss: -1.9746
Epoch 17, Avg Loss: -1.9849
Epoch 18, Avg Loss: -1.9926
Epoch 19, Avg Loss: -2.0036
Epoch 20, Avg Loss: -2.0076
Epoch 21, Avg Loss: -2.0101
Epoch 22, Avg Loss: -2.0184
Epoch 23, Avg Loss: -2.0240
Epoch 24, Avg Loss: -2.0287
Epoch 25, Avg Loss: -2.0448
Epoch 26, Avg Loss: -2.0421
Epoch 27, Avg Loss: -2.0552
Epoch 28, Avg Loss: -2.0624
Epoch 29, Avg Loss: -2.0621
Epoch 30, Avg Loss: -2.0709


In [49]:
torch.save(vae.state_dict(), 'vae.pth')
torch.save(rnn.state_dict(), 'rnn.pth')

In [50]:
# generate_video function credit: opus 5
def generate_video(vae, rnn, device=DEVICE, z_dim=Z_DIM, width=3840, height=2160, fps=15, frames=150, output_path='imagination_4k.mp4'):
    vae.eval()
    rnn.eval()
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    with torch.no_grad():
        z = torch.zeros(1, 1, z_dim).to(device) + torch.randn(1, 1, z_dim, device=device) * 0.1
        hidden = None
        for i in range(frames):
            steering = 0.4 * np.sin(i / 6.0) * np.cos(i / 12.0)
            action = torch.tensor([[[steering, 0.0]]], dtype=torch.float32).to(device)
            pi, mu, sigma, hidden = rnn(z, action, hidden)

            probs = F.softmax(pi.squeeze(), dim=0)  
            mode_idx = torch.multinomial(probs, 1).item()
            selected_mu = mu[0, 0, mode_idx]  
            noise = torch.randn_like(selected_mu) * 0.05
            z_pred = selected_mu + noise
            z = z_pred.unsqueeze(0).unsqueeze(0)

            img = vae.decode(z.squeeze(0)).squeeze(0)
            img_np = (img.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
            img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
            img_4k = cv2.resize(img_bgr, (width, height), interpolation=cv2.INTER_CUBIC)
            cv2.putText(img_4k, f'STEER: {steering:.2f}', (100, 150),
                        cv2.FONT_HERSHEY_SIMPLEX, 3, (255, 255, 255), 5)
            cv2.putText(img_4k, f'LATENT SPACE: {z_dim}D | DEVICE: {DEVICE_NAME}', (100, 250),
                        cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 3)
            out.write(img_4k)
    out.release()
generate_video(vae, rnn)